<a href="https://colab.research.google.com/github/andreydeps/estudo-ci-ncia-de-dados/blob/main/TCC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [27]:
!pip install spacy
!pip install pdfplumber

import pandas as pd
import numpy as np
import spacy
import pdfplumber

In [28]:
#conectando ao drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [29]:
#modelo de linguagem em português
!python -m spacy download pt_core_news_lg
nlp = spacy.load("pt_core_news_lg")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 568.2/568.2 MB 935.5 kB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('pt_core_news_lg')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [30]:
#leitura do pdf
caminho = "/content/drive/MyDrive/Repositório TCC/Documentação/PPA/ppa_2016_2019.pdf"

with pdfplumber.open(caminho) as pdf:
    texto = ""
    for pagina in pdf.pages:
        texto += pagina.extract_text()

Estrutura da documentação ppa 2016-2019



```
PROGRAMA [código] [nome] ÓRGÃO RESPONSÁVEL [órgão]
OBJETIVO [texto]
JUSTIFICATIVA [texto]
PÚBLICO-ALVO [texto]
CUSTO DO PROGRAMA / META FINANCEIRA / tabela de SUBAÇÃO...\```



In [31]:
# Extrai código, nome, órgão, objetivo e justificativa de cada programa,
# usando OBJETIVO/JUSTIFICATIVA/PÚBLICO-ALVO como marcadores de início/fim de cada campo
padrao = re.compile(
    r"PROGRAMA\s+(?P<codigo>\d{4})\s+(?P<nome>.*?)\s+"
    r"ÓRGÃO RESPONSÁVEL\s+(?P<orgao>.*?)\s+"
    r"OBJETIVO\s+(?P<objetivo>.*?)\s+"
    r"JUSTIFICATIVA\s+(?P<justificativa>.*?)\s+"
    r"PÚBLICO-ALVO",
    re.DOTALL
)

matches = list(padrao.finditer(texto))
print(f"Achei {len(matches)} blocos com programa vinculado")

df = pd.DataFrame([m.groupdict() for m in matches])
df.head()

Achei 107 blocos com programa vinculado


,codigo,nome,orgao,objetivo,justificativa
0,0810,Comunicação do Poder Executivo,Gabinete do Governador do Estado,Fazer prevalecer o direito do cidadão de ser i...,Respeitar o direito de acesso a informação que...
1,0825,Formação de Gestores Públicos,Secretaria de Estado da Fazenda,Desenvolver cursos ciclo longo e cursos ciclo ...,Necessidade de gestores capazes de atender as ...
2,0830,Modernização da Administração Fazendária,Secretaria de Estado da Fazenda,"Modernizar a administração pública, visando au...",Necessidade de incrementar a qualidade dos ser...
3,0850,Gestão de Pessoas,Secretaria de Estado da Administração,Modernizar os instrumentos de gestão na área d...,Necessidade de modernizar os atuais instrument...
4,0855,Saúde Ocupacional,Secretaria de Estado da Administração,"Realizar ações de normatização, coordenação, s...",Atender a Lei nº 14.609 de 07 de janeiro de 20...


In [32]:
# Remove duplicatas antes de qualquer filtro, pra não contar o mesmo programa duas vezes
df = df.drop_duplicates(subset=["codigo", "objetivo"], keep="first")
df["codigo"].value_counts().sort_values(ascending=False).head(10)

,count
codigo,
0810,1
0825,1
0830,1
0850,1
0855,1
0870,1
0900,1
0990,1
0100,1


In [33]:
# Códigos de programas de política industrial, identificados manualmente na tabela-índice do PPA
codigos_industria = ["0200", "0230", "0310", "0342", "0346"]
df_industria = df[df["codigo"].isin(codigos_industria)]
df_industria

,codigo,nome,orgao,objetivo,justificativa
19,0200,Competitividade e Excelência Econômica,Secretaria de Estado do Desenvolvimento\nEconô...,Fomentar a atividade produtiva no estado e pro...,O ambiente externo é favorável ao comércio nac...
25,0230,"INOVAR - Fomento à Pesquisa, ao Desenvolviment...",Secretaria de Estado do Desenvolvimento\nEconô...,Aplicar os recursos destinados à pesquisa cien...,"Buscar a ampliação, formação, readequação de c..."
28,0310,Agronegócio Competitivo,Secretaria de Estado da Agricultura e da Pesca,Incrementar a base de conhecimentos científico...,A economia mundial exige melhoria da produtivi...
33,0342,Revitalização da Economia Catarinense - PREC,Secretaria de Estado do Desenvolvimento\nEconô...,Promover o desenvolvimento econômico sustentáv...,As micro e pequenas empresas e os empreendedor...
34,0346,Tecnologia e Inovação para o Desenvolvimento S...,Secretaria de Estado do Desenvolvimento\nEconô...,Promover e incentivar a tecnologia e a inovaçã...,O tímido envolvimento de empresas privadas em ...


Teste agora com todos os PPAS


In [38]:
def processa_ppa(caminho, periodo, duas_colunas=False, regex=None):
    regex = regex or padrao
    with pdfplumber.open(caminho) as pdf:
        texto = ""
        for pagina in pdf.pages:
            if duas_colunas:
                largura, altura = pagina.width, pagina.height
                esquerda = pagina.crop((0, 0, largura/2, altura)).extract_text() or ""
                direita = pagina.crop((largura/2, 0, largura, altura)).extract_text() or ""
                texto += esquerda + "\n" + direita
            else:
                texto += pagina.extract_text() or ""

    matches = list(regex.finditer(texto))
    print(f"{periodo}: achei {len(matches)} blocos com programa vinculado")

    if len(matches) == 0:
        print(f"AVISO: {periodo} não bateu com o padrão atual")
        return pd.DataFrame(columns=["codigo", "nome", "orgao", "objetivo", "justificativa", "periodo"])

    df = pd.DataFrame([m.groupdict() for m in matches])
    df["periodo"] = periodo
    df = df.drop_duplicates(subset=["codigo", "objetivo"], keep="first")
    df_industria = df[df["codigo"].isin(codigos_industria)]
    return df_industria

In [35]:
# Configuração
caminhos = {
    "2008-2011": "/content/drive/MyDrive/Repositório TCC/Documentação/PPA/ppa_2008_2011.pdf",
    "2012-2015": "/content/drive/MyDrive/Repositório TCC/Documentação/PPA/ppa_2012_2015.pdf",
    "2016-2019": "/content/drive/MyDrive/Repositório TCC/Documentação/PPA/ppa_2016_2019.pdf",
    "2020-2023": "/content/drive/MyDrive/Repositório TCC/Documentação/PPA/ppa_2020_2023.pdf",
    "2024-2027": "/content/drive/MyDrive/Repositório TCC/Documentação/PPA/ppa_2024_2027.pdf",
}

codigos_industria = ["0200", "0230", "0310", "0342", "0346"]

duas_colunas = {"2020-2023"}

# Roda pra todos os PPAs
resultados = []
for periodo, caminho in caminhos.items():
    resultados.append(processa_ppa(caminho, periodo, duas_colunas=(periodo in duas_colunas)))

df_todos = pd.concat(resultados, ignore_index=True)

2008-2011: achei 0 blocos com programa vinculado
AVISO: 2008-2011 não bateu com o padrão atual, precisa investigar separado
2012-2015: achei 115 blocos com programa vinculado
2016-2019: achei 107 blocos com programa vinculado
2020-2023: achei 0 blocos com programa vinculado
AVISO: 2020-2023 não bateu com o padrão atual, precisa investigar separado
2024-2027: achei 0 blocos com programa vinculado
AVISO: 2024-2027 não bateu com o padrão atual, precisa investigar separado


In [36]:
# Regex de estruturação por programa
padrao = re.compile(
    r"PROGRAMA\s+(?P<codigo>\d{4})\s+(?P<nome>.*?)\s+"
    r"ÓRGÃO RESPONSÁVEL\s+(?P<orgao>.*?)\s+"
    r"OBJETIVO\s+(?P<objetivo>.*?)\s+"
    r"JUSTIFICATIVA\s+(?P<justificativa>.*?)\s+"
    r"PÚBLICO-ALVO",
    re.DOTALL
)

In [37]:
# Diagnóstico dos que não bateram
for periodo in ["2008-2011", "2020-2023", "2024-2027"]:
    with pdfplumber.open(caminhos[periodo]) as pdf:
        if periodo in duas_colunas:
            texto_periodo = ""
            for pagina in pdf.pages:
                largura, altura = pagina.width, pagina.height
                esquerda = pagina.crop((0, 0, largura/2, altura)).extract_text() or ""
                direita = pagina.crop((largura/2, 0, largura, altura)).extract_text() or ""
                texto_periodo += esquerda + "\n" + direita
        else:
            texto_periodo = "".join(pagina.extract_text() or "" for pagina in pdf.pages)

    print(f"=== {periodo} ===")
    print("OBJETIVO:", "OBJETIVO" in texto_periodo)
    print("PROGRAMA:", "PROGRAMA" in texto_periodo)
    print("JUSTIFICATIVA:", "JUSTIFICATIVA" in texto_periodo)
    print(texto_periodo[:800])
    print()

=== 2008-2011 ===
OBJETIVO: False
PROGRAMA: True
JUSTIFICATIVA: False
ESTADO DE SANTA CATARINA
ANEXO ÚNICO
Plano Plurianual 2008 - 2011
2011/2011 2011/2011
PROGRAMA/AÇÕES/SUBAÇÕES PRODUTO UNIDADE FÍSICO
OGE OF
0100 ProPav Rural Tipo Finalístico
Órgão Responsável Secretaria de Estado da Infraestrutura Horizonte Temporal Contínuo
Objetivo Viabilizar a pavimentação de acessos e ruas das comunidades rurais.
Justificativa Integrar as comunidades rurais à malha viária do estado.
Público Alvo Moradores das localidades beneficiadas
Indicador Km de rodovia pavimentada
0057 Terraplenagem/pavimentação/OAE/supervisão de rodovias Rodovia pavimentada
010210 Pavimentação trecho São Martinho - Vargem do Cedro km 14 100.000 0
010867 Pavimentação asfáltica do acesso à Universidade km 1 400.000 0
Federal Fronteira Sul
0151 Terraplanagem e pavimentação de acessos e trechos de O

=== 2020-2023 ===
OBJETIVO: True
PROGRAMA: True
JUSTIFICATIVA: True
a) Programas Temá
b) Programas de
Serviços; e
II – o Anexo I

In [40]:
padrao_2008 = re.compile(
    r"(?P<codigo>\d{4})\s+(?P<nome>.*?)\s+Tipo\s+\S+\s+"
    r"Órgão Responsável\s+(?P<orgao>.*?)\s+Horizonte Temporal\s+\S+\s+"
    r"Objetivo\s+(?P<objetivo>.*?)\s+"
    r"Justificativa\s+(?P<justificativa>.*?)\s+"
    r"Público Alvo",
    re.DOTALL
)

KeyboardInterrupt: 

In [41]:
df_2008 = processa_ppa(caminhos["2008-2011"], "2008-2011", regex=padrao_2008)
df_2008

2008-2011: achei 36 blocos com programa vinculado


,codigo,nome,orgao,objetivo,justificativa,periodo


In [42]:
pos = texto_periodo.find("OBJETIVO")
print(texto_periodo[max(0, pos-300):pos+1000])

 de Abastecimento do Estado de Santa Catarina S.A. 0 0 1.450.000 1.450.000 5.800.000
12Anexo I
PLANO PLURIANUAL 2024 - 2027
ORÇAMENTO FISCAL - PROGRAMAS DE GESTÃO, MANUTENÇÃO E DE SERVIÇOS AO ESTADO
PROGRAMA 0810 Comunicação do Poder Executivo UNIDADE RESPONSÁVEL Secretaria de Estado da Comunicação
OBJETIVO Fazer prevalecer o direito do cidadão de ser informado e o dever do homem público de informar.
JUSTIFICATIVA Respeitar o direito de acesso à informação que todo cidadão possui referente as ações governamentais, assim respeitando também um dos novos conceitos do
novo serviço público que é a accountability.
PÚBLICO-ALVO Cidadãos, investidores, turistas e consumidores
INDICADOR UNIDADE FONTE POLARIDADE VALOR DATA META AO FINAL DO PPA
REFERÊNCIA APURAÇÃO
0219 Campanhas de carácter social, informativa e institucional unidade SECOM/Sistema Maior Melhor 20,00 24/07/2023 80,00
SCP
0220 Publicações legais na mídia impressa unidade SECOM/SC Maior Melhor 22,00 24/07/2023 88,00
CUSTO DO PROGRAM

In [43]:
padrao = re.compile(
    r"PROGRAMA\s+(?P<codigo>\d{4})\s+(?P<nome>.*?)\s+"
    r"(?:ÓRGÃO|UNIDADE) RESPONSÁVEL\s+(?P<orgao>.*?)\s+"
    r"OBJETIVO\s+(?P<objetivo>.*?)\s+"
    r"JUSTIFICATIVA\s+(?P<justificativa>.*?)\s+"
    r"PÚBLICO-ALVO",
    re.DOTALL
)

In [44]:
df_2024 = processa_ppa(caminhos["2024-2027"], "2024-2027")

2024-2027: achei 104 blocos com programa vinculado


In [46]:
with pdfplumber.open(caminhos["2008-2011"]) as pdf:
    texto_2008 = "".join(pagina.extract_text() or "" for pagina in pdf.pages)

matches_2008 = list(padrao_2008.finditer(texto_2008))
df_2008_completo = pd.DataFrame([m.groupdict() for m in matches_2008])
df_2008_completo = df_2008_completo.drop_duplicates(subset=["codigo", "objetivo"], keep="first")

pd.set_option("display.max_colwidth", None)
df_2008_completo[["codigo", "nome"]]

codigo  \
0    2008   
1    0057   
2    0057   
3    0057   
4    0158   
5    0058   
6    0189   
7    0066   
8    0068   
9    0064   
10   0191   
11   0846   
12   2008   
13   0477   
14   0117   
15   0039   
16   0028   
17   0040   
18   0878   
19   0080   
20   0008   
21   2008   
22   0332   
23   0006   
24   0003   
25   0052   
26   0053   
27   0051   
28   0302   
29   0281   
30   0317   
31   0324   
32   0125   
33   0140   
34   0232   
35   0227   

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                 

In [48]:
codigos_industria = {
    "2008-2011": ["0200", "0210", "0230", "0250"],  # 0250 (Inclusão Digital) a confirmar com o pezzini
    "2012-2015": [...],
    "2016-2019": ["0200", "0230", "0310", "0342", "0346"],
    "2020-2023": [...],
    "2024-2027": [...],
}